<a href="https://colab.research.google.com/github/emilyrgarman/ml3finalproject_triage/blob/main/model_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
! git clone https://github.com/emilyrgarman/ml3finalproject_triage.git

Cloning into 'ml3finalproject_triage'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 23 (delta 6), reused 15 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 8.73 MiB | 11.92 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

SEED = 42

Loading in and merging train and test sets with chief complaints text data

In [5]:
train = pd.read_csv('/content/ml3finalproject_triage/train.csv')
test = pd.read_csv('/content/ml3finalproject_triage/test.csv')
complaints = pd.read_csv('/content/ml3finalproject_triage/chief_complaints.csv')
history = pd.read_csv('/content/ml3finalproject_triage/patient_history.csv')

In [6]:
train_df = train.merge(complaints[['patient_id', 'chief_complaint_raw']], on='patient_id', how='left')
train_df = train_df.merge(history, on='patient_id', how='left')
test_df  = test.merge(complaints[['patient_id', 'chief_complaint_raw']], on='patient_id', how='left')
test_df = test_df.merge(history, on='patient_id', how='left')

In [7]:
train_df.head()

,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,hx_hypothyroidism,hx_hyperthyroidism,hx_hiv,hx_coagulopathy,hx_immunosuppressed,hx_pregnant,hx_substance_use_disorder,hx_coronary_artery_disease,hx_stroke_prior,hx_peripheral_vascular_disease
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,1,1,0,0,1,0,0,0,0,0
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,1,0,0,0,1,0,1,1,0,0
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,0,1,1,0,0,0,0,1,1,1
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,0,0,1,0,1,0,0,0,0,1
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,1,1,0,1,1,0,1,1,1,0


In [8]:
train_df.columns

Index(['patient_id', 'site_id', 'triage_nurse_id', 'arrival_mode',
       'arrival_hour', 'arrival_day', 'arrival_month', 'arrival_season',
       'shift', 'age', 'age_group', 'sex', 'language', 'insurance_type',
       'transport_origin', 'pain_location', 'mental_status_triage',
       'chief_complaint_system', 'num_prior_ed_visits_12m',
       'num_prior_admissions_12m', 'num_active_medications',
       'num_comorbidities', 'systolic_bp', 'diastolic_bp',
       'mean_arterial_pressure', 'pulse_pressure', 'heart_rate',
       'respiratory_rate', 'temperature_c', 'spo2', 'gcs_total', 'pain_score',
       'weight_kg', 'height_cm', 'bmi', 'shock_index', 'news2_score',
       'disposition', 'ed_los_hours', 'triage_acuity', 'chief_complaint_raw',
       'hx_hypertension', 'hx_diabetes_type2', 'hx_diabetes_type1',
       'hx_asthma', 'hx_copd', 'hx_heart_failure', 'hx_atrial_fibrillation',
       'hx_ckd', 'hx_liver_disease', 'hx_malignancy', 'hx_obesity',
       'hx_depression', 'hx_anxiet

Preprocessing and train/validation split

In [9]:
NUMERIC_COLS = ['age', 'num_active_medications', 'num_comorbidities',
                'systolic_bp', 'diastolic_bp', 'heart_rate',
                'respiratory_rate', 'temperature_c', 'spo2',
                'gcs_total', 'pain_score', 'bmi',
                'num_prior_ed_visits_12m', 'num_prior_admissions_12m']
CATEG_COLS   = ['arrival_mode', 'sex', 'transport_origin',
                'pain_location', 'mental_status_triage']
DUMMY_COLS   = ['hx_hypertension', 'hx_diabetes_type2', 'hx_diabetes_type1',
                'hx_asthma', 'hx_copd', 'hx_heart_failure', 'hx_atrial_fibrillation',
                'hx_ckd', 'hx_liver_disease', 'hx_malignancy', 'hx_obesity',
                'hx_depression', 'hx_anxiety', 'hx_dementia', 'hx_epilepsy',
                'hx_hypothyroidism', 'hx_hyperthyroidism', 'hx_hiv', 'hx_coagulopathy',
                'hx_immunosuppressed', 'hx_pregnant', 'hx_substance_use_disorder',
                'hx_coronary_artery_disease', 'hx_stroke_prior',
                'hx_peripheral_vascular_disease']

ID_COLS      = ['patient_id', 'site_id', 'triage_nurse_id']
DISCARDED_COLS = ['arrival_hour', 'arrival_day',
                  'arrival_month', 'arrival_season', 'shift',
                  'age_group', 'disposition', 'shock_index',
                  'ed_los_hours', 'news2_score',
                  'mean_arterial_pressure', 'pulse_pressure',
                  'chief_complaint_system',
                  'language', 'insurance_type', 'weight_kg',
                  'height_cm']

TARGET_COL   = 'triage_acuity'
TEXT_COL     = 'chief_complaint_raw'
TABULAR_COLS = NUMERIC_COLS + CATEG_COLS + DUMMY_COLS

for col in CATEG_COLS:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col]  = le.transform(test_df[col].astype(str))

imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_tab      = scaler.fit_transform(imputer.fit_transform(train_df[TABULAR_COLS]))
X_tab_test = scaler.transform(imputer.transform(test_df[TABULAR_COLS]))

# Shift labels 1–5 → 0–4 for CrossEntropyLoss
y = train_df[TARGET_COL].values - 1

# Train/val split
X_tr, X_val, y_tr, y_val, idx_tr, idx_val = train_test_split(
    X_tab, y, np.arange(len(y)),
    test_size=0.25, stratify=y, random_state=SEED
)

texts_all  = train_df[TEXT_COL].fillna('').values
texts_tr   = texts_all[idx_tr]
texts_val  = texts_all[idx_val]
texts_test = test_df[TEXT_COL].fillna('').values

print(f'\nTrain: {len(y_tr)} | Val: {len(y_val)} | Test: {len(test_df)}')


Train: 60000 | Val: 20000 | Test: 20000


ANN (tabular)

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Model definition ───────────────────────────────────────────────
class TabularANN(nn.Module):
    def __init__(self, input_dim, num_classes=5, hidden=[128, 64], dropout=0.3):
        super().__init__()
        layers = []
        in_d = input_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # raw logits

# ── DataLoaders ────────────────────────────────────────────────────
def make_loader(X, y=None, batch_size=256, shuffle=False):
    Xt = torch.tensor(X, dtype=torch.float32)
    if y is not None:
        yt = torch.tensor(y.squeeze(), dtype=torch.long)
        return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)
    return DataLoader(TensorDataset(Xt), batch_size=batch_size)

tr_loader  = make_loader(X_tr,  y_tr,  shuffle=True)
val_loader = make_loader(X_val, y_val)

# ── Training ───────────────────────────────────────────────────────
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS  = 30

ann       = TabularANN(input_dim=X_tr.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(ann.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss, best_state = float('inf'), None

for epoch in range(1, EPOCHS + 1):
    ann.train()
    for Xb, yb in tr_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(ann(Xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()

    # Validation
    ann.eval()
    val_loss, preds, trues = 0, [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            logits = ann(Xb.to(DEVICE))
            val_loss += criterion(logits, yb.to(DEVICE)).item()
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())

    val_loss /= len(val_loader)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = ann.state_dict()

    if epoch % 5 == 0:
        from sklearn.metrics import f1_score
        f1 = f1_score(trues, preds, average='macro')
        print(f'Epoch {epoch:3d} | Val Loss: {val_loss:.4f} | Val Macro-F1: {f1:.4f}')

ann.load_state_dict(best_state)
print('Training complete. Best val loss:', round(best_val_loss, 4))

# ── Get probabilities for fusion later ────────────────────────────
def get_probs(model, X):
    model.eval()
    loader = make_loader(X)
    probs  = []
    with torch.no_grad():
        for (Xb,) in loader:
            logits = model(Xb.to(DEVICE))
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(probs)

ann_val_probs  = get_probs(ann, X_val)   # save for late fusion
ann_test_probs = get_probs(ann, X_tab_test)

Epoch   5 | Val Loss: 0.4380 | Val Macro-F1: 0.8282
Epoch  10 | Val Loss: 0.4109 | Val Macro-F1: 0.8412
Epoch  15 | Val Loss: 0.4020 | Val Macro-F1: 0.8453
Epoch  20 | Val Loss: 0.3941 | Val Macro-F1: 0.8481
Epoch  25 | Val Loss: 0.3910 | Val Macro-F1: 0.8517
Epoch  30 | Val Loss: 0.3900 | Val Macro-F1: 0.8501
Training complete. Best val loss: 0.3897


ClinicalBERT (text)

In [13]:
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

BERT_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_LEN    = 64

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

# ── Dataset ────────────────────────────────────────────────────────
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], max_length=MAX_LEN,
                        padding='max_length', truncation=True,
                        return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ── Model ──────────────────────────────────────────────────────────
class ClinicalBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes=5, dropout=0.1):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.clf     = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out     = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        cls_emb = out.last_hidden_state[:, 0, :]  # [CLS] token embedding
        logits  = self.clf(self.dropout(cls_emb))
        return logits, cls_emb  # return both for early fusion later

# ── DataLoaders ────────────────────────────────────────────────────
tr_text_ds  = TextDataset(texts_tr,  y_tr.squeeze())
val_text_ds = TextDataset(texts_val, y_val.squeeze())

tr_text_loader  = DataLoader(tr_text_ds,  batch_size=64, shuffle=True)
val_text_loader = DataLoader(val_text_ds, batch_size=64)

# ── Training ───────────────────────────────────────────────────────
BERT_EPOCHS = 1  # fine-tuning doesn't need many epochs

bert_clf       = ClinicalBERTClassifier(BERT_MODEL).to(DEVICE)
bert_optimizer = torch.optim.AdamW(bert_clf.parameters(), lr=2e-5, weight_decay=1e-2)
bert_criterion = nn.CrossEntropyLoss()
bert_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(bert_optimizer, T_max=BERT_EPOCHS)

best_bert_loss, best_bert_state = float('inf'), None

for epoch in range(1, BERT_EPOCHS + 1):
    bert_clf.train()
    for batch in tqdm(tr_text_loader, desc=f'Epoch {epoch}'):
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels    = batch['labels'].to(DEVICE)

        bert_optimizer.zero_grad()
        logits, _ = bert_clf(input_ids, attn_mask)
        loss      = bert_criterion(logits, labels)
        loss.backward()
        bert_optimizer.step()
    bert_scheduler.step()

    # Validation
    bert_clf.eval()
    val_loss, preds, trues = 0, [], []
    with torch.no_grad():
        for batch in val_text_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            labels    = batch['labels'].to(DEVICE)
            logits, _ = bert_clf(input_ids, attn_mask)
            val_loss += bert_criterion(logits, labels).item()
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(labels.cpu().numpy())

    val_loss /= len(val_text_loader)
    if val_loss < best_bert_loss:
        best_bert_loss  = val_loss
        best_bert_state = bert_clf.state_dict()

    f1 = f1_score(trues, preds, average='macro')
    print(f'Epoch {epoch} | Val Loss: {val_loss:.4f} | Val Macro-F1: {f1:.4f}')

bert_clf.load_state_dict(best_bert_state)
print('Training complete. Best val loss:', round(best_bert_loss, 4))

# ── Get probabilities and embeddings for fusion ────────────────────
def get_bert_probs_and_embeddings(model, texts, labels=None):
    model.eval()
    ds     = TextDataset(texts, labels)
    loader = DataLoader(ds, batch_size=32)
    probs, embeddings = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            logits, cls_emb = model(input_ids, attn_mask)
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            embeddings.append(cls_emb.cpu().numpy())
    return np.vstack(probs), np.vstack(embeddings)

bert_val_probs,  bert_val_embs  = get_bert_probs_and_embeddings(bert_clf, texts_val)
bert_test_probs, bert_test_embs = get_bert_probs_and_embeddings(bert_clf, texts_test)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Val Loss: 0.0020 | Val Macro-F1: 0.9975
Training complete. Best val loss: 0.002


^^ Note: takes 15 min to run as is (1 epoch, 64 max len, 64 batch)

Fusion

In [14]:
# ── Late Fusion (average probabilities) ───────────────────────────
late_val_probs  = (ann_val_probs  + bert_val_probs)  / 2
late_test_probs = (ann_test_probs + bert_test_probs) / 2

late_val_preds  = late_val_probs.argmax(axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print('=== Late Fusion Validation ===')
print(classification_report(y_val.squeeze(), late_val_preds,
                             target_names=[f'Acuity {i+1}' for i in range(5)]))

# ── Early Fusion (concatenate BERT embeddings + tabular features) ──
# bert_val_embs is (N, 768), X_val is (N, num_tabular_features)
# concatenate along feature dimension
early_val_X  = np.concatenate([bert_val_embs,  X_val],       axis=1)
early_test_X = np.concatenate([bert_test_embs, X_tab_test],  axis=1)

class EarlyFusionHead(nn.Module):
    def __init__(self, input_dim, num_classes=5, hidden=[256, 128], dropout=0.3):
        super().__init__()
        layers = []
        in_d = input_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# need embeddings for train split too
bert_tr_probs, bert_tr_embs = get_bert_probs_and_embeddings(bert_clf, texts_tr)
early_tr_X = np.concatenate([bert_tr_embs, X_tr], axis=1)

tr_early_loader  = make_loader(early_tr_X,  y_tr,  shuffle=True)
val_early_loader = make_loader(early_val_X, y_val)

INPUT_DIM    = early_tr_X.shape[1]  # 768 + num_tabular_features
fusion_head  = EarlyFusionHead(input_dim=INPUT_DIM).to(DEVICE)
fus_optimizer = torch.optim.AdamW(fusion_head.parameters(), lr=1e-3, weight_decay=1e-4)
fus_criterion = nn.CrossEntropyLoss()
fus_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(fus_optimizer, T_max=30)

best_fus_loss, best_fus_state = float('inf'), None

for epoch in range(1, 31):
    fusion_head.train()
    for Xb, yb in tr_early_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        fus_optimizer.zero_grad()
        loss = fus_criterion(fusion_head(Xb), yb)
        loss.backward()
        fus_optimizer.step()
    fus_scheduler.step()

    fusion_head.eval()
    val_loss, preds, trues = 0, [], []
    with torch.no_grad():
        for Xb, yb in val_early_loader:
            logits = fusion_head(Xb.to(DEVICE))
            val_loss += fus_criterion(logits, yb.to(DEVICE)).item()
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())

    val_loss /= len(val_early_loader)
    if val_loss < best_fus_loss:
        best_fus_loss  = val_loss
        best_fus_state = fusion_head.state_dict()

    if epoch % 5 == 0:
        f1 = f1_score(trues, preds, average='macro')
        print(f'Epoch {epoch:3d} | Val Loss: {val_loss:.4f} | Val Macro-F1: {f1:.4f}')

fusion_head.load_state_dict(best_fus_state)

print('\n=== Early Fusion Validation ===')
fusion_head.eval()
preds, trues = [], []
with torch.no_grad():
    for Xb, yb in val_early_loader:
        preds.extend(fusion_head(Xb.to(DEVICE)).argmax(1).cpu().numpy())
        trues.extend(yb.numpy())

print(classification_report(trues, preds,
                             target_names=[f'Acuity {i+1}' for i in range(5)]))

# ── Compare all models ─────────────────────────────────────────────
for name, p in [('ANN only',      ann_val_probs.argmax(1)),
                ('BERT only',     bert_val_probs.argmax(1)),
                ('Late Fusion',   late_val_preds),
                ('Early Fusion',  preds)]:
    f1 = f1_score(y_val.squeeze(), p, average='macro')
    print(f'{name:<20} Macro-F1: {f1:.4f}')

=== Late Fusion Validation ===
              precision    recall  f1-score   support

    Acuity 1       1.00      0.99      1.00       806
    Acuity 2       1.00      1.00      1.00      3360
    Acuity 3       1.00      1.00      1.00      7230
    Acuity 4       1.00      1.00      1.00      5755
    Acuity 5       1.00      1.00      1.00      2849

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000

Epoch   5 | Val Loss: 0.0007 | Val Macro-F1: 0.9995
Epoch  10 | Val Loss: 0.0012 | Val Macro-F1: 0.9985
Epoch  15 | Val Loss: 0.0006 | Val Macro-F1: 0.9994
Epoch  20 | Val Loss: 0.0015 | Val Macro-F1: 0.9983
Epoch  25 | Val Loss: 0.0007 | Val Macro-F1: 0.9994
Epoch  30 | Val Loss: 0.0007 | Val Macro-F1: 0.9994

=== Early Fusion Validation ===
              precision    recall  f1-score   support

    Acuity 1       1.00      1.00      1.00       806
    Acuity 2       1.00   

Eval

In [18]:
import numpy as np

# Cost matrix: cost[true][pred] — penalizes by distance between acuity levels
cost_matrix = np.array([[0, 1, 2, 3, 4],
                         [1, 0, 1, 2, 3],
                         [2, 1, 0, 1, 2],
                         [3, 2, 1, 0, 1],
                         [4, 3, 2, 1, 0]])

def cost_sensitive_score(y_true, y_pred, cost_matrix):
    total_cost = sum(cost_matrix[t][p] for t, p in zip(y_true, y_pred))
    # normalize by worst possible cost
    worst_cost = sum(cost_matrix[t].max() for t in y_true)
    return 1 - (total_cost / worst_cost)  # higher is better

score = cost_sensitive_score(y_val.squeeze(), late_val_preds, cost_matrix)
print(f'Cost-sensitive score: {score:.4f}')

Cost-sensitive score: 0.9999


there must be an issue here lol

In [15]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_tr, y_tr)
print('Dummy F1:', f1_score(y_val.squeeze(), dummy.predict(X_val), average='macro'))

Dummy F1: 0.10620639001101725


In [20]:
print(train_df['chief_complaint_raw'].sample(10).to_string())

45200           acute angle closure glaucoma with vomiting
59787                          minor joint ache with fever
36557                   dietary advice diabetic with fever
52727          significant blunt thoracic trauma, constant
4625                 severe dyspnoea at rest, intermittent
14991                   renal colic moderate with vomiting
31000    electrolyte imbalance symptomatic, worsening w...
3243            suicidal ideation without plan, for 2 days
41464                traumatic amputation with diaphoresis
53236       routine BP monitoring, worsening with movement


In [ ]:
## may want to do something like this, and then see how different it is
## to determine how much these words are carrying the predictions
import re
SEVERITY_WORDS = ['mild', 'moderate', 'severe','critical', 'minor', 'major', 'actively']
train_df['cc_stripped'] = train_df['chief_complaint_raw'].apply(
    lambda x: re.sub('|'.join(SEVERITY_WORDS), '', x, flags=re.IGNORECASE).strip()
)